# Heart Disease

In [ ]:

import time
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import dvgate as dg
import heart_loader as H

SEED, PATH, PILOT, N_BOOT = 99, "datasets/Heart_Disease datos/heart", 200, 400
T0 = time.time()

dg.THR["k_context"] = 1
print(f"umbrales: {dg.THR}\n  ^ k_context bajado a 1 para n=4 (parámetro de escala)")

## 1 — Carga

In [ ]:
D = H.load(PATH, seed=SEED)
X, y, src, P = D["X"], D["y"], D["src"], D["players"]
TR, VA, TE, NOM = D["idx_tr"], D["idx_va"], D["idx_te"], D["nombres"]
v, v0, idxp = dg.make_game(X, y, src, TR, VA, P, cap=None, seed=SEED)
print(f"\nv(vacío)={v0:.4f}  v(N)={v(P):.4f}  excedente={v(P) - v0:.4f}  "
      f"reparto medio={(v(P) - v0) / len(P):.4f}")

## 2 — Panel

In [ ]:
t = time.time()
pan = dg.run_panel(X, y, src, TR, VA, TE, P, v, D["meta"], SEED, PILOT, PILOT)
MIN_PANEL = (time.time() - t) / 60
print(pan["tabla"].to_string(index=False))
print(f"\nconcluimos: {pan['veredicto']} — {pan['motivo']}  ({MIN_PANEL * 60:.0f}s)")
print("\nno aplicable")
DAN = pan["daño"].copy()
DAN.index = NOM
print("\n" + DAN[["n_eval", "prev", "lift_out", "lift_self", "lift_cross",
                  "ret", "se_ret", "sin_poder", "tipo"]].round(3).to_string())

## 3 — Shapley exacto

In [ ]:
t = time.time()
SC = {frozenset(S): (None if not S else dg._fit(
        X[np.concatenate([idxp[p] for p in S])],
        y[np.concatenate([idxp[p] for p in S])], SEED).predict_proba(X[VA])[:, 1])
      for k in range(len(P) + 1) for S in combinations(P, k)}
PHI = dg.exact_shapley(P, lambda S: v0 if not S else
                       float(average_precision_score(y[VA], SC[frozenset(S)])))

yv, rng, bs = y[VA], np.random.default_rng(SEED), []
for _ in range(N_BOOT):
    i = rng.integers(0, len(yv), len(yv))
    if len(np.unique(yv[i])) < 2:
        continue
    bs.append(dg.exact_shapley(P, lambda S: float(yv[i].mean()) if not S else
              float(average_precision_score(yv[i], SC[frozenset(S)][i]))))
SE = {p: float(np.std([b[p] for b in bs], ddof=1)) for p in P}
LO = dg.loo(P, v)
MIN_CARO = (time.time() - t) / 60

VAL = pd.DataFrame({"phi": pd.Series(PHI), "se": pd.Series(SE), "loo": pd.Series(LO)})
VAL[["ret", "ret_cross", "tipo", "dañina"]] = pan["daño"][["ret", "ret_cross", "tipo", "dañina"]]
VAL["phi_2se"] = VAL.phi + dg.THR["sigma_rule"] * VAL.se
VAL.index = NOM
print(VAL.round(4).to_string())
print(f"\neficiencia: suma - (v(N)-v(vacío)) = {sum(PHI.values()) - (v(P) - v0):+.1e}"
      f"  |  16 coaliciones + {len(bs)} remuestras en {MIN_CARO * 60:.0f}s")

## 4 — close_loop

In [ ]:
CL, guiada = dg.close_loop(X, y, src, TR, TE, P, PHI, SE, D["meta"], SEED, n_random=4)
print(CL[["k", "contexto", "auroc", "ap", "d_ap", "d_ap_lo", "d_ap_hi"]].round(4).to_string())
print(f"\nregla phi + 2SE < 0 marca: {[NOM[p] for p in guiada] or 'nadie (salida vacía)'}")

## 5 — correlacion con el exacto

In [ ]:
CONC = pd.DataFrame({
    "phi_exacto": VAL.phi, "phi+2SE": VAL.phi_2se, "phi<0": VAL.phi < 0,
    "panel_dañina": VAL["dañina"], "panel_tipo": VAL.tipo, "loo<0": VAL.loo < 0})
print(CONC.round(4).to_string())
ac = int((CONC["panel_dañina"] == CONC["phi<0"]).sum())
print(f"\ncmparamos panel y Shapley exacto: {ac}/{len(P)} fuentes\n"
      f"conclusion del panel: {pan['veredicto']} — {pan['motivo']}\n"
      f"panel {MIN_PANEL * 60:.0f}s frente a {MIN_CARO * 60:.0f}s del reparto exacto: "
      f"con n=4 el cálculo sale bieb, y por eso aquí no hay nada que "
      f"validarr.\ntotal {(time.time() - T0) / 60:.1f} min")

for nom, obj in [("heart_valores", VAL), ("heart_dano", DAN),
                 ("heart_cierre_bucle", CL), ("heart_concordancia", CONC)]:
    obj.to_csv(f"{nom}.csv")
print("\n ficheros: heart_valores.csv heart_dano.csv heart_cierre_bucle.csv "
      "heart_concordancia.csv")